[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aledamelio/ML_Pavia/blob/main/04_Nonlinear_Classification.ipynb)

# Non-Linear Classification

As seen throughout these lessons, certain datasets are not linearly separable. For these datasets, linear classifiers are often insufficient to build useful classification models.

This limitation has been recognized since the early days of AI. One of the most famous examples is the XOR Problem, highlighted by Marvin Minsky and Seymour Papert in 1969. In their book Perceptrons, Minsky and Papert demonstrated that a simple Exclusive OR (XOR) logic gate cannot be modeled by a single-layer perceptron.

![XOR Problem](images/XORp.png)

This example demonstrates how—unlike the OR gate—the XOR function generates a dataset that is not linearly separable.

Next, we will explore techniques for extending linear classifiers to handle non-linear decision boundaries, as well as examine models that are inherently non-linear.

## Feature Mapping

- Is it possible to learn non-linear decision boundaries?

- $\textbf{Feature Mapping}$ or $\textbf{Feature Expansion}$: the core idea is to create a non-linear combination of the original features in order to project data points into a higher-dimensional space where (hopefully) the data becomes linearly separable.

Consider the following non-linearly separable dataset:

In [ ]:
from sklearn.datasets import make_circles, make_moons
import matplotlib.pyplot as plt
from pandas import DataFrame
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

def plot_decision_regions(X, y, classifier, resolution=0.02, feature_map=None):

    # setup marker generator and color map
    markers = ('s', 'x', 'o', '^', 'v')
    colors = ('red', 'blue', 'lightgreen', 'gray', 'cyan')
    cmap = ListedColormap(colors[:len(np.unique(y))])

    # plot the decision surface
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))
    
    if feature_map == None:
        Z = classifier.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    else:
        Z = classifier.predict(feature_map.transform(np.array([xx1.ravel(), xx2.ravel()]).T))
    Z = Z.reshape(xx1.shape)
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    # plot class examples
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(x=X[y == cl, 0], 
                    y=X[y == cl, 1],
                    alpha=0.8, 
                    c=colors[idx],
                    marker=markers[idx], 
                    label=cl, 
                    edgecolor='black')
        

# generate 2d classification dataset
X, y = make_circles(factor=0.2, n_samples=200, noise=0.07, random_state=123)

# plot data
cm_bright = ListedColormap(['red', 'blue'])
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cm_bright, edgecolors='k')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')

plt.show()


We map the dataset from the original two-dimensional space, to a three-dimensional space through the following projection:

$$\phi\left(x_{1}, x_{2}\right) = \left(x_{1}, x_{2}, x_{1}^{2}+x_{2}^{2}\right) = \left(z_{1}, z_{2}, z_{3}\right)$$

We define the $\phi\left(x_{1}, x_{2}\right)$ function in Python:

In [ ]:
import numpy as np

class my_feature_map():
    
    def transform(self, X):
        n_samples = X.shape[0]
        Z = np.zeros([X.shape[0], 3])
        for i in range(n_samples):
            z = np.array([X[i,0], X[i,1], (X[i,0]**2 + X[i,1]**2)])
            Z[i,:] = z

        return Z

phi = my_feature_map()
Z = phi.transform(X)

fig = plt.figure(figsize=(15, 6))
ax = fig.add_subplot(1,2,1)

# plot data
cm_bright = ListedColormap(['red', 'blue'])
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cm_bright, edgecolors='k')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

ax = fig.add_subplot(1,2,2, projection='3d')
ax.scatter(Z[:, 0], Z[:, 1], Z[:,2], c=y, cmap=cm_bright, edgecolors='k')
ax.set_xlabel('21')
ax.set_ylabel('z2')
ax.set_zlabel('z3')
ax.view_init(elev=20., azim=32)

plt.show()

We can now train a linear classifier (e.g. a SVM) on the original and mapped data:

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_fm = LogisticRegression(random_state=123, solver='lbfgs')
lr_fm.fit(Z, y)

w = lr_fm.coef_.flatten()
b = lr_fm.intercept_.flatten()
print('w=',w,'b=',b)

# create x,y
xx, yy = np.meshgrid(np.linspace(-1,1), np.linspace(-1,1))

# calculate corresponding z
boundary = (-w[0] * xx - w[1] * yy - b) * 1. /w[2]

fig = plt.figure(figsize=(15, 6))
ax = fig.add_subplot(1,2,1)
# plot data
cm_bright = ListedColormap(['red', 'blue'])
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cm_bright, edgecolors='k')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

ax = fig.add_subplot(1,2,2, projection='3d')
ax.scatter(Z[:, 0], Z[:, 1], Z[:,2], c=y, cmap=cm_bright, edgecolors='k')
ax.plot_surface(xx, yy, boundary, alpha = .3)
ax.view_init(elev=20., azim=32)
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_zlabel('$z_3$')

plt.show()

In [ ]:
lr = LogisticRegression(random_state=123, solver='lbfgs')
lr.fit(X, y)

plt.figure(figsize=(16,5))
plt.subplot(1,2,1)
plot_decision_regions(X, y, classifier=lr)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.title('SVM Original')

plt.subplot(1,2,2)
plot_decision_regions(X, y, feature_map=phi, classifier=lr_fm)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.title('SVM Feature Mapping')

plt.show()


## Polynomial Feature Mapping in Python

A general way to expand features is to use polynomials of degree $n$. Polynomial feature expansion maps data from the original space into a higher-dimensional space. Specifically, assuming we start from a two-dimensional space (as in the previous example) with a generic feature vector:

$$\mathbf{x} = [x_1, x_2] \in \mathbb{R}^2$$

a polynomial map of degree $n=2$ generates the transformed vector:

$$\phi\left(x_{1}, x_{2}\right) = \left(1, x_1, x_2, x_1^2, x_1x_2, x_2^2\right) \in \mathbb{R}^6$$

This polynomial transformation includes:

- The original features ($x_1$ and $x_2$)
- A bias term ($1$)
- Interaction features, i.e., the products of all possible feature combinations of degree 2 ($x_1x_2$)
- Polynomial features up to degree $n$ ($x_1^2$ and $x_2^2$)

We will implement polynomial feature expansion in Python on the *make_moons* dataset using the *PolynomialFeatures* class.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
import pandas as pd

pd.set_option("display.precision", 2)

# generate 2d classification dataset
X, y = make_moons(n_samples=200, noise=0.07, random_state=123)
scaler = StandardScaler()
X = scaler.fit_transform(X)

quadratic = PolynomialFeatures(degree=2)
X_quad = quadratic.fit_transform(X)

print(pd.DataFrame(X[:5,:], columns=['x1', 'x2']))
print(' ')
print(pd.DataFrame(X_quad[:5,:],  columns=quadratic.get_feature_names_out(['x1', 'x2'])))

Let's train a linear SVM on the dataset mapped with grade 2 polynomial.

In [ ]:
lr = LogisticRegression(random_state=123, solver='lbfgs')
lr.fit(X_quad, y)

plot_decision_regions(X, y, feature_map=quadratic, classifier=lr)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('SVM with 2nd degree polynimal features')
plt.show()

As we can observe, the decision boundary plotted in the original space is non-linear. However, a polynomial of degree $2$ does not seem sufficient to separate the two classes in the transformed space. Let's try using a higher-degree polynomial ($d=3$).

In [ ]:
cubic = PolynomialFeatures(degree=3)
X_cubic = cubic.fit_transform(X)

print(pd.DataFrame(X[:5,:], columns=['x1', 'x2']))
print(' ')
print(pd.DataFrame(X_cubic[:5,:],  columns=cubic.get_feature_names_out(['x1', 'x2'])))

lr = LogisticRegression(random_state=123, solver='lbfgs')
lr.fit(X_cubic, y)

plot_decision_regions(X, y, feature_map=cubic, classifier=lr)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('SVM with 3rd degree polynimal features')
plt.show()

The _PolynomialFeatures_ class allows you to choose whether to use bias and polynomial features through attributes:

- _include_bias_ (default to _True_ )
- _interaction_only_ (default to _False_)

The following code shows the decision boundaries that are obtained using only the interaction features on 3 non-linearly separable datasets (circles, moons and XOR)

In [ ]:
quadratic_i = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

#dataset circles
X_circles, y_circles = make_circles(factor=0.5, n_samples=200, noise=0.07, random_state=123)
X_circles = scaler.fit_transform(X_circles)

#dataset XOR
np.random.seed(123)
X_xor = np.random.randn(200, 2)
y_xor = np.logical_xor(X_xor[:, 0] > 0, X_xor[:, 1] > 0)
X_xor = scaler.fit_transform(X_xor)
y_xor = np.where(y_xor, 1, 0)

#dataset moons
X_moons, y_moons = make_moons(n_samples=200, noise=0.1, random_state=123)
X_moons = scaler.fit_transform(X_moons)

#feature mapping usando solo le feature di interazione
X_circles_int = quadratic_i.fit_transform(X_circles)
X_xor_int = quadratic_i.fit_transform(X_xor)
X_moons_int = quadratic_i.fit_transform(X_moons)

#fitting SVM lineare sulle feature mappate 
lr_circles = LogisticRegression(random_state=123, solver='lbfgs')
lr_circles.fit(X_circles_int, y_circles)
lr_xor = LogisticRegression(random_state=123, solver='lbfgs')
lr_xor.fit(X_xor_int, y_xor)
lr_moons = LogisticRegression(random_state=123, solver='lbfgs')
lr_moons.fit(X_moons_int, y_moons)

#plot decision boundaries
plt.figure(figsize=(16,5))
plt.subplot(1,3,1)
plot_decision_regions(X_circles, y_circles, feature_map=quadratic_i, classifier=lr_circles)
plt.legend(loc='upper left')
plt.tight_layout()
plt.title('SVM interaction features Circles')

plt.subplot(1,3,2)
plot_decision_regions(X_xor, y_xor, feature_map=quadratic_i, classifier=lr_xor)
plt.legend(loc='upper left')
plt.tight_layout()
plt.title('SVM interaction features Xor')

plt.subplot(1,3,3)
plot_decision_regions(X_moons, y_moons, feature_map=quadratic_i, classifier=lr_moons)
plt.legend(loc='upper left')
plt.tight_layout()
plt.title('SVM interaction features Moons')
plt.show()

Note that interaction features alone are not sufficient to linearly separate the classes for the *circles* and *moons* datasets, though they are for the XOR dataset.

By using only the interaction features on a two-dimensional dataset, we can plot the points in the expanded space; the interaction feature is, in fact, given solely by the product of $x_1$ and $x_2$. This yields a mapping to a 3-dimensional space:

In [ ]:
# create x,y
xx, yy = np.meshgrid(np.linspace(-3,3), np.linspace(-4,4))

w = lr_xor.coef_.flatten()
b = lr_xor.intercept_.flatten()

# calculate corresponding z
boundary = (-w[0] * xx - w[1] * yy - b) * 1. /w[2]

fig = plt.figure(figsize=(15, 6))
ax = fig.add_subplot(1,2,1)
# plot data
plot_decision_regions(X_xor, y_xor, feature_map=quadratic_i, classifier=lr_xor)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('SVM with interaction features only')

ax = fig.add_subplot(1,2,2, projection='3d')
ax.scatter(X_xor_int[:, 0], X_xor_int[:, 1], X_xor_int[:,2], c=y_xor, cmap=cm_bright, edgecolors='k')
ax.plot_surface(xx, yy, boundary, alpha = .3)
ax.view_init(elev=20., azim=50)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$x_3$')
plt.show()

- The feature expansion technique allows us to generalize each linear classifier to non-linear classification problems.

- Increasing the degree of polynomial leads to increasingly complex decision-making boundaries.

In [ ]:
from sklearn.linear_model import Perceptron, LogisticRegression

#dataset moons
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=123)

cubic = PolynomialFeatures(degree=3)
X_moons = scaler.fit_transform(X_moons)
X_cubic = cubic.fit_transform(X_moons)

lr = LogisticRegression(C=10, random_state=123, max_iter=10000)
lr.fit(X_cubic, y_moons)

ppn = Perceptron(eta0=0.01, random_state=123)
ppn.fit(X_cubic, y_moons)

#plot decision boundaries
plt.figure(figsize=(16,5))
plt.subplot(1,2,1)
plot_decision_regions(X_moons, y_moons, feature_map=cubic, classifier=lr)
plt.legend(loc='upper left')
plt.tight_layout()
plt.title('LR with 3rd degree polynimal')

plt.subplot(1,2,2)
plot_decision_regions(X_moons, y_moons, feature_map=cubic, classifier=ppn)
plt.legend(loc='upper left')
plt.tight_layout()
plt.title('Perceptron with 3rd degree polynimal')
plt.show()

# k-Nearest Neighbors (k-NN) Classification

- A different approach compared to those seen so far
- Does not learn discriminative functions (lines, hyperplanes, etc.)
- Works by "memorizing" the entire dataset (no optimization, no iterative learning procedures)
- Belongs to the class of $\textit{non-parametric}$ methods

The algorithm can be summarized as follows:

- Choose a number $k$ and a distance metric (e.g., Euclidean distance)
- Find the $k$ nearest neighbors to the sample you want to classify
- Assign the class label by majority vote

<img src="images/03_23.png" width="600">

- Very simple and intuitive classifier
- Directly extensible to multi-class problems
- Requires storing the entire dataset (impractical for large-scale datasets)

## k-NN in Pyhton

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X, y)

plot_decision_regions(X, y, classifier=knn)

plt.xlabel('petal length [standardized]')
plt.ylabel('petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

### Distance measures

The $\textit{sklearn.neighbors.DistanceMetric}$ class implements different metric types:

<img src="images/d_metrics.png" width="600">

In [ ]:
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

iris = datasets.load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

sc = StandardScaler()
X = sc.fit_transform(X)

distances = ['euclidean', 'manhattan', 'chebyshev']

plt.figure(figsize=(16,5))
for i,d in enumerate(distances):
    knn = KNeighborsClassifier(n_neighbors=5, metric=d)
    knn.fit(X, y)
    plt.subplot(1,3,i+1)
    plot_decision_regions(X, y, classifier=knn)
    plt.xlabel('petal length [standardized]')
    plt.ylabel('petal width [standardized]')
    plt.legend(loc='upper left')
    plt.title('k-NN - distance:' + d)
    plt.tight_layout()
plt.show()

### The number of neighbors $k$

In [ ]:
ks = np.arange(1,12,2)

plt.figure(figsize=(16,10))
for i,k in enumerate(ks):
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X, y)
    plt.subplot(2,3,i+1)
    plot_decision_regions(X, y, classifier=knn)
    plt.xlabel('petal length [standardized]')
    plt.ylabel('petal width [standardized]')
    plt.legend(loc='upper left')
    plt.title('k-NN - k=' + str(k))
    plt.tight_layout()
plt.show()